# 🔮 EXODUS-SPECULUM - Colab Pipeline Template

> Système de transformation vidéo immobilière en clone 3D

---

## Instructions d'utilisation

1. **Connecter Google Drive** (Cell 1)
2. **Configurer les variables** (Cell 2)
3. **Installer les dépendances** (Cell 3)
4. **Exécuter les tests de validation** (Cells 7-8)

In [ ]:
# ============================================================
# 📁 Cell 1: Montage Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive monté avec succès")

In [ ]:
# ============================================================
# ⚙️ Cell 2: Configuration Globale
# ============================================================
import os

# Mode de qualité: eclaireur | conquerant | souverain
TURBO_MODE = "conquerant"

# Format de sortie: HORIZONTAL | VERTICAL | SQUARE
OUTPUT_FORMAT = "HORIZONTAL"

# Chemins
WORKSPACE_ROOT = "/content/exodus_workspace"
SHARED_RESOURCES = "/content/drive/MyDrive/EXODUS_SHARED_RESOURCES"
AI_MODELS_PATH = os.path.join(SHARED_RESOURCES, "AI_MODELS")
ASSETS_HUB_PATH = os.path.join(SHARED_RESOURCES, "ASSETS_HUB")
OUTPUT_PATH = "/content/output"

# Créer les dossiers de travail
for path in [WORKSPACE_ROOT, OUTPUT_PATH]:
    os.makedirs(path, exist_ok=True)

print(f"✅ Configuration chargée")
print(f"   Mode: {TURBO_MODE}")
print(f"   Format: {OUTPUT_FORMAT}")
print(f"   Workspace: {WORKSPACE_ROOT}")

In [ ]:
# ============================================================
# 📦 Cell 3: Installation Dépendances Core
# ============================================================
%%capture install_output
!pip install torch torchvision --extra-index-url https://download.pytorch.org/whl/cu118
!pip install opencv-python-headless pillow tqdm pyyaml

print("✅ Dépendances core installées")

In [ ]:
# ============================================================
# 🎮 Cell 4: Installation Blender (bpy)
# ============================================================
%%capture blender_install
!pip install bpy==4.0.0

# Vérification
try:
    import bpy
    print(f"✅ Blender installé: {bpy.app.version_string}")
except ImportError as e:
    print(f"❌ Erreur installation Blender: {e}")

In [ ]:
# ============================================================
# 🔑 Cell 5: Configuration API Keys
# ============================================================
from google.colab import userdata
import os

# Récupérer la clé Gemini depuis les secrets Colab
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    print("✅ GEMINI_API_KEY configurée")
except Exception as e:
    print(f"⚠️ GEMINI_API_KEY non trouvée dans les secrets")
    print("   → Aller dans: Clé à gauche > Secrets > Ajouter GEMINI_API_KEY")

In [ ]:
# ============================================================
# 🏗️ Cell 6: Création Structure Ressources Partagées
# ============================================================
import os

SHARED_RESOURCES = "/content/drive/MyDrive/EXODUS_SHARED_RESOURCES"

# Structure des dossiers
STRUCTURE = {
    "AI_MODELS": [
        "depth_anything_v2",
        "yolov8",
        "sam"
    ],
    "ASSETS_HUB": [
        "furniture",
        "decorations",
        "test"
    ]
}

print("🏗️ Vérification structure EXODUS_SHARED_RESOURCES...")
print("=" * 50)

for category, subdirs in STRUCTURE.items():
    category_path = os.path.join(SHARED_RESOURCES, category)
    
    if not os.path.exists(category_path):
        os.makedirs(category_path, exist_ok=True)
        print(f"📁 Créé: {category}/")
    else:
        print(f"✅ Existe: {category}/")
    
    for subdir in subdirs:
        subdir_path = os.path.join(category_path, subdir)
        if not os.path.exists(subdir_path):
            os.makedirs(subdir_path, exist_ok=True)
            print(f"   📁 Créé: {category}/{subdir}/")
        else:
            print(f"   ✅ Existe: {category}/{subdir}/")

print("=" * 50)
print("✅ Structure vérifiée")

In [ ]:
# ============================================================
# 🧠 Cell 7: Test Chargement Modèle IA depuis Drive
# ============================================================
import os
import time
import torch

# Chemins ressources partagées
SHARED_RESOURCES = "/content/drive/MyDrive/EXODUS_SHARED_RESOURCES"
AI_MODELS_PATH = os.path.join(SHARED_RESOURCES, "AI_MODELS")
DEPTH_MODEL_PATH = os.path.join(AI_MODELS_PATH, "depth_anything_v2")

print("=" * 60)
print("TEST: Chargement Depth Anything V2 depuis Drive")
print("=" * 60)

# Vérifier existence du dossier
if not os.path.exists(DEPTH_MODEL_PATH):
    print(f"⚠️ Dossier non trouvé: {DEPTH_MODEL_PATH}")
    print("📥 Création de la structure...")
    os.makedirs(DEPTH_MODEL_PATH, exist_ok=True)
    print("⚠️ ATTENTION: Vous devez télécharger le modèle manuellement:")
    print("   https://huggingface.co/depth-anything/Depth-Anything-V2-Large")
    print(f"   → Placer dans: {DEPTH_MODEL_PATH}/")
    MODEL_AVAILABLE = False
else:
    # Chercher le fichier de poids
    model_files = [f for f in os.listdir(DEPTH_MODEL_PATH) 
                   if f.endswith(('.pth', '.safetensors', '.pt'))]
    
    if model_files:
        print(f"✅ Modèle trouvé: {model_files[0]}")
        MODEL_FILE = os.path.join(DEPTH_MODEL_PATH, model_files[0])
        MODEL_AVAILABLE = True
        
        # Test de chargement avec mesure de latence
        print("\n⏱️ Test de latence chargement...")
        start_time = time.time()
        
        try:
            # Chargement des poids (sans le modèle complet pour ce test)
            state_dict = torch.load(MODEL_FILE, map_location='cpu')
            load_time = time.time() - start_time
            
            print(f"✅ Chargement réussi en {load_time:.2f} secondes")
            print(f"   Taille state_dict: {len(state_dict)} clés")
            
            # Libérer mémoire
            del state_dict
            torch.cuda.empty_cache()
            
            # Évaluation latence
            if load_time < 10:
                print("🚀 LATENCE EXCELLENTE (< 10s)")
            elif load_time < 30:
                print("✅ LATENCE ACCEPTABLE (< 30s)")
            else:
                print("⚠️ LATENCE ÉLEVÉE - Considérer cache local")
                
        except Exception as e:
            print(f"❌ Erreur chargement: {e}")
            MODEL_AVAILABLE = False
    else:
        print(f"⚠️ Aucun fichier modèle dans: {DEPTH_MODEL_PATH}")
        print("   Fichiers attendus: .pth, .safetensors, .pt")
        MODEL_AVAILABLE = False

print("=" * 60)

In [ ]:
# ============================================================
# 🔗 Cell 8: Test Blender Library Linking depuis Drive
# ============================================================
import bpy
import os
import time

ASSETS_HUB = "/content/drive/MyDrive/EXODUS_SHARED_RESOURCES/ASSETS_HUB"
TEST_ASSET = os.path.join(ASSETS_HUB, "test_asset.blend")

print("=" * 60)
print("TEST: Blender Library Linking depuis Drive")
print("=" * 60)

# Créer le dossier ASSETS_HUB si inexistant
if not os.path.exists(ASSETS_HUB):
    os.makedirs(ASSETS_HUB, exist_ok=True)
    print(f"📁 Créé: {ASSETS_HUB}")

# Créer un asset de test si inexistant
if not os.path.exists(TEST_ASSET):
    print("📦 Création asset de test...")
    
    # Nouvelle scène propre
    bpy.ops.wm.read_factory_settings(use_empty=True)
    
    # Créer un cube stylisé (Ghost Proxy prototype)
    bpy.ops.mesh.primitive_cube_add(size=1, location=(0, 0, 0))
    cube = bpy.context.active_object
    cube.name = "GhostProxy_TestCube"
    
    # Ajouter metadata custom property
    cube["asset_type"] = "furniture"
    cube["asset_category"] = "test"
    cube["is_ghost_proxy"] = True
    
    # Sauvegarder
    bpy.ops.wm.save_as_mainfile(filepath=TEST_ASSET)
    print(f"✅ Asset créé: {TEST_ASSET}")

# Test de Library Linking
print("\n⏱️ Test de linking...")
start_time = time.time()

# Reset pour le test
bpy.ops.wm.read_factory_settings(use_empty=True)

try:
    # Link depuis le fichier externe
    with bpy.data.libraries.load(TEST_ASSET, link=True) as (data_from, data_to):
        print(f"   Objets disponibles: {list(data_from.objects)}")
        data_to.objects = list(data_from.objects)
    
    link_time = time.time() - start_time
    
    # Ajouter les objets linkés à la scène
    for obj in data_to.objects:
        if obj is not None:
            bpy.context.collection.objects.link(obj)
            print(f"   ✅ Linké: {obj.name}")
            
            # Vérifier les custom properties
            if obj.get("is_ghost_proxy"):
                print(f"      → Ghost Proxy détecté!")
                print(f"      → Type: {obj.get('asset_type', 'unknown')}")
    
    print(f"\n✅ LINKING RÉUSSI en {link_time:.2f} secondes")
    
    # Évaluation
    if link_time < 5:
        print("🚀 LATENCE EXCELLENTE (< 5s)")
    elif link_time < 15:
        print("✅ LATENCE ACCEPTABLE (< 15s)")
    else:
        print("⚠️ LATENCE ÉLEVÉE - Le système Ghost Proxy pourrait être lent")
        
except Exception as e:
    print(f"❌ ERREUR LINKING: {e}")

print("=" * 60)